In [1]:
from pathlib import Path

import numpy as np
import dask.distributed as dd
import scanpy as sc
import anndata as ad
import h5py
import dask

from collections import Counter
import pandas as pd
from tqdm import tqdm
import dask
import time
from dask.distributed import Client, LocalCluster

import dask
import re 
sc.logging.print_header()

/opt/conda/lib/python3.12/site-packages/session_info2/__init__.py:124: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  and (v := getattr(pkg, "__version__", None))
/tmp/ipykernel_19612/1794876524.py:19: RuntimeWarning: Failed to import dependencies for application/vnd.jupyter.widget-view+json representation. (ModuleNotFoundError: No module named 'ipywidgets')
  sc.logging.print_header()


Package,Version
numpy,1.26.4
dask,2025.2.0
scanpy,1.11.1
anndata,0.11.4
h5py,3.13.0
pandas,2.2.3
tqdm,4.67.0
distributed,2025.2.0
Component,Info
Python,"3.12.10 | packaged by conda-forge | (main, Apr 10 2025, 22:21:13) [GCC 13.3.0]"


In [2]:
use_gpu = False

In [3]:
if use_gpu:
    import rapids_singlecell as rsc
    SPARSE_CHUNK_SIZE = 1_000_000
    from dask_cuda import LocalCUDACluster

    import rmm
    import cupy as cp
    from cupyx.scipy import spx

    from rmm.allocators.cupy import rmm_cupy_allocator

    def set_mem():
        rmm.reinitialize(managed_memory=True)
        cp.cuda.set_allocator(rmm_cupy_allocator)
    gpus = "0" # comma separated like "0,1,2" for 3 gpus
    cluster = LocalCUDACluster(CUDA_VISIBLE_DEVICES=gpus)
    dask.array.register_chunk_type(spx.csr_matrix)
else:

    SPARSE_CHUNK_SIZE = 100_000
    cluster = LocalCluster(n_workers=16)
    #cluster = LocalCluster(n_workers=16, threads_per_worker=2)

In [4]:
client = Client(cluster)
if use_gpu:
    client.run(set_mem)
    mod = rsc
else:
    mod = sc

In [5]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 16
Total threads: 32,Total memory: 300.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46416,Workers: 16
Dashboard: http://127.0.0.1:8787/status,Total threads: 32
Started: Just now,Total memory: 300.00 GiB
Comm: tcp://127.0.0.1:41481,Total threads: 2
Dashboard: http://127.0.0.1:45838/status,Memory: 18.75 GiB
Nanny: tcp://127.0.0.1:43048,


In [15]:
#get the drugs we have genes for: matching drug to gene pickle file 
df = pd.read_pickle('final_dataset.pkl')
drugs = df.compound.to_list()
escaped = [re.escape(d) for d in drugs]
drug_pattern = r'(' + '|'.join(escaped) + r')'

In [16]:
%%time
adatas = []
all_highly_variable_genes = []
for i in tqdm(range(14)):
    id = str(i + 1)
    PATH = f"data/plate{id}_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad"

    with h5py.File(PATH, "r") as f:
        adata = ad.AnnData(
            obs=ad.io.read_elem(f["obs"]),
            var=ad.io.read_elem(f["var"]),
        )
        adata.X = ad.experimental.read_elem_as_dask(
            f["X"], chunks=(SPARSE_CHUNK_SIZE, adata.shape[1])
        )

    if use_gpu:
        rsc.get.anndata_to_GPU(adata)

    # 100m filtering
    pass_filter_mask = adata.obs["pass_filter"] == "full"
    adata = adata[pass_filter_mask, :].copy()

    #now select for only drugs of interest and WT cells 
    drugconc = adata.obs['drugname_drugconc'].astype(str)
    mask_drug = drugconc.str.contains(drug_pattern, case=False, na=False)
    mask_5uM = drugconc.str.contains('5.0', case=False, na=False)
    mask_dmsotf = drugconc.eq("[('DMSO_TF', 0.0, 'uM')]") 
    mask = (mask_drug & mask_5uM) | mask_dmsotf
    
    print(f"Total cells before filter: {adata.n_obs}")
    print(f"Cells matching drug + 5.0:   {mask.sum()}")
    adata = adata[mask, :].copy()

    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=5000)

    highly_variable_genes = set(adata.var_names[adata.var["highly_variable"]])
    all_highly_variable_genes.append(highly_variable_genes)
    adatas.append(adata)

## select the genes appears more than two plates
gene_counts = Counter(gene for genes in all_highly_variable_genes for gene in genes)
selected_genes = {gene for gene, count in gene_counts.items() if count > 2}

for i in tqdm(range(14)):
    id = str(i + 1)

    common_genes = list(set(adata.var_names) & selected_genes)
    adata = adatas[i][:, common_genes].copy()

    output_path = f"data/plate{id}_filtered_preprocessed_{'gpu' if use_gpu else 'cpu'}.h5ad"
    adata.write_h5ad(output_path)

  0%|                                                                                                                              | 0/14 [00:00<?, ?it/s]<timed exec>:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


Total cells before filter: 5299930
Cells matching drug + 5.0:   171297


/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 22.06 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
  7%|████████▎                                                                                                            | 1/14 [04:09<53:59, 249.19s/it]<timed exec>:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


Total cells before filter: 7687184
Cells matching drug + 5.0:   245537


/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 31.78 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 14%|████████████████▍                                                                                                  | 2/14 [09:56<1:01:21, 306.75s/it]<timed exec>:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


...

Total cells before filter: 8106500
Cells matching drug + 5.0:   1169186


/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 36.98 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 93%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 13/14 [2:20:49<12:33, 753.34s/it]<timed exec>:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


Total cells before filter: 6201077
Cells matching drug + 5.0:   2342032


/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 34.09 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
  0%|                                                                                                                              | 0/14 [00:00<?, ?it/s]/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 22.08 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
  7%|████████▍                      

CPU times: user 1h 50min 2s, sys: 22min 40s, total: 2h 12min 43s
Wall time: 7h 13min 41s


In [27]:
#merge into a single file 
data_dict = {}
for i in range(14):
    data_dict.update({f"plate_{i+1}": f"data/plate{i+1}_filtered_preprocessed_{'gpu' if use_gpu else 'cpu'}.h5ad"})
ad.experimental.concat_on_disk(
    data_dict,
    f'data/plate_merged_{"gpu" if use_gpu else "cpu"}.h5ad',
    label='plate',
)

In [6]:
#read in the file 
with h5py.File(f"data/plate_merged_{'gpu' if use_gpu else 'cpu'}.h5ad", "r") as f:
    adata = ad.AnnData(
        obs=ad.io.read_elem(f["obs"]),
        var=ad.io.read_elem(f["var"]),
    )
    adata.X = ad.experimental.read_elem_as_dask(
        f["X"], chunks=(SPARSE_CHUNK_SIZE, adata.shape[1])
    )

In [7]:
adata

AnnData object with n_obs × n_vars = 19538423 × 6627
    obs: 'sample', 'gene_count', 'tscp_count', 'mread_count', 'drugname_drugconc', 'drug', 'cell_line', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'pass_filter', 'cell_name', 'plate'

In [8]:
#calculate PCA
mod.pp.pca(adata, n_comps=300, mask_var=None)

In [9]:
adata

AnnData object with n_obs × n_vars = 19538423 × 6627
    obs: 'sample', 'gene_count', 'tscp_count', 'mread_count', 'drugname_drugconc', 'drug', 'cell_line', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'pass_filter', 'cell_name', 'plate'
    uns: 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [10]:
# 1) pull out just the two metadata columns (we’ll need their index)
obs = adata.obs[['drugname_drugconc','cell_line']]

# 2) keep only groups with ≥100 cells
obs_filt = obs.groupby(['drugname_drugconc','cell_line']).filter(lambda df: len(df) >= 100)

# 3) sample 200 cells from each group
#    group_keys=False makes apply return a flat DataFrame with the original index
sampled = (
    obs_filt
    .groupby(['drugname_drugconc','cell_line'], group_keys=False)
    .apply(lambda df: df.sample(n=100, random_state=42))
)

# 4) get the cell-barcodes (index) and subset your AnnData
idx = sampled.index
adata_sub = adata[idx, :].copy()

print(f"Original cells: {adata.n_obs:,}")
print(f"Kept groups:    {len(sampled.groupby(['drugname_drugconc','cell_line']))}")
print(f"Sampled cells:  {adata_sub.n_obs:,}")

/tmp/ipykernel_19612/2049314493.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_filt = obs.groupby(['drugname_drugconc','cell_line']).filter(lambda df: len(df) >= 100)
/tmp/ipykernel_19612/2049314493.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['drugname_drugconc','cell_line'], group_keys=False)
/tmp/ipykernel_19612/2049314493.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupi

Original cells: 19,538,423


/tmp/ipykernel_19612/2049314493.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(f"Kept groups:    {len(sampled.groupby(['drugname_drugconc','cell_line']))}")


Kept groups:    8866
Sampled cells:  886,600


In [11]:
adata_sub

AnnData object with n_obs × n_vars = 886600 × 6627
    obs: 'sample', 'gene_count', 'tscp_count', 'mread_count', 'drugname_drugconc', 'drug', 'cell_line', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'pass_filter', 'cell_name', 'plate'
    uns: 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [20]:
Y = adata_sub.obs[['drugname_drugconc','cell_line']]
Y.to_csv('sampled_metadata.csv')

In [17]:
import dask.dataframe as dd
import pyarrow as pa
import pyarrow.parquet as pq
import dask.array as da
import shutil, os
from dask.array import to_npy_stack

In [18]:
X = adata_sub.obsm['X_pca']  
X

dask.array<getitem, shape=(886600, 300), dtype=float32, chunksize=(99685, 300), chunktype=numpy.ndarray>

In [19]:
outdir = "X_pca_npy"

# Remove any previous dump
if os.path.exists(outdir):
    shutil.rmtree(outdir)

# Write the chunks out
to_npy_stack(outdir, X, axis=0)

/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 15.01 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [22]:
# each .npy in X_pca_npy/ becomes one chunk along axis-0
X = da.from_npy_stack("X_pca_npy")

print(X.shape)   # (886600, 300)
print(X.chunks)  # e.g. ((99685, 99685, …), (300,))

(886600, 300)
((99685, 99685, 99685, 99685, 99685, 99685, 99685, 99685, 89120), (300,))


In [25]:
import h5py

In [27]:
# 2) Pick a uniform chunk shape: first chunk along each axis
chunk_rows = X.chunks[0][0]    # 99685
chunk_cols = X.chunks[1][0]    # 300

# 3) Remove any existing output
fname = "X_pca.h5"
if os.path.exists(fname):
    os.remove(fname)

# 4) Create the HDF5 dataset with that chunk layout
with h5py.File(fname, "w") as f:
    dset = f.create_dataset(
        "X_pca", 
        shape=X.shape, 
        dtype=X.dtype, 
        chunks=(chunk_rows, chunk_cols),
        compression="gzip"       # or None/"lzf"/etc.
    )

    # 5) Stream each Dask chunk into place
    offset = 0
    for block_len in X.chunks[0]:
        # bring only that block into memory
        block = X[offset: offset + block_len, :].compute()
        dset[offset: offset + block_len, :] = block
        offset += block_len